In [ ]:
#imports
import pandas as pd
import nibabel as nib
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## Config
subsample = False #subsample == True: with plasma p-tau217; subsample == False: without plasma p-tau217
save_plots = True 
savename = 'unet_spat_res'

# Specify the directory for input freesurfer mask and true tau-PET images
taupet_root = " "
mask_root = " "

In [ ]:
# load the evaluation file (output from examples/evaluate_unet.py) and test set dataframe

df_eval = pd.read_csv('../outputs/eval/evaluation_results.csv')
df_test = pd.read_csv('../datasets/df_test.csv')

if subsample:
    df_test = df_test.dropna(subset='plasma_ptau217')
    
df_eval = df_eval.merge(df_test)

# Specify the hue order and corresponding datasets. Example from article Figure 4.
hue_order = ['bf2','bf1','mcgill','ucsf','adni','a4','berkeley','oasis','avid_a04','avid_a05','avid_a08',
             'avid_llcf','avid_lzax']
if subsample:
    hue_order = ['bf2','bf1','mcgill','ucsf','adni','a4']

orderdict = {'bf2':0,'adni':2,'avid_a05':1.5,'bf1':6,'mcgill':4,'ucsf':5,'a4':1,
             'berkeley':3,'oasis':3,'avid_a04':3,'avid_a08':0,'avid_llcf':.5,'avid_lzax':3}

df_eval['study_order'] = df_eval['study'].map(orderdict)
df_eval = df_eval.sort_values(by='study_order')

In [ ]:
# Load freesurfer regions and select the ones to use for regional spatial correlation evaluation
fs_text = pd.read_csv('../src/utils/fs.txt',sep=' ')
fs_text = fs_text.rename(columns={'0':'number','Unknown':'region'})
fs_list = fs_text.iloc[371:443]

In [ ]:
## Calculate the suvr in true and synthetic tau-PET for the different freesurfer regions for all test cases
dfs_fs = []

for name in df_eval['file_names']:
    #load the images and freesurfer masks
    mri_mask = nib.load(mask_root + name + '/jademarec.nii.gz')
    mri_mask_data = mri_mask.get_fdata()
    synt_img = nib.load('../outputs/synthetic_test_scans/' + name + '.nii.gz')
    synt_img_data = synt_img.get_fdata()
    true_img = nib.load(taupet_root + name + '/smoothed_image.nii.gz')
    true_img_data = true_img.get_fdata()
    
    #add voxel values and divide by number of voxels for each regions
    real_sum_values,gen_sum_values, nr_vox =  [], [], []
    for nr in fs_list['number']:
        mask = mri_mask_data == nr
        nr_vox.append(np.sum(mask))
        real_sum_values.append(np.sum(true_img_data[mask]))
        gen_sum_values.append(np.sum(synt_img_data[mask]))

    nr_vox = np.array(nr_vox)
    region_df = pd.DataFrame([np.array(real_sum_values)/nr_vox,
                             np.array(gen_sum_values)/nr_vox]).T
    
    # save result in dataframe
    region_df.columns = ['true_suvr','synt_suvr']
    region_df['region'] = fs_list['region'].reset_index(drop=True)
    dfs_fs.append(region_df)
    print('done')

In [ ]:
#remove regions that did not exist or is unknown
df = dfs_fs[0].dropna() 
rgs = df['region']
rgs = [rg for rg in rgs if 'unknown' not in rg]

# rearrange dfs for easier calculations
dfs_reg = []
for region in rgs:
    dfs_reg.append(pd.DataFrame(columns=['true_suvr_' + region,'synt_suvr']))
                   
for df in dfs_fs:
    for region,df2 in zip(rgs,dfs_reg):
        use = df[df['region'] == region]
        df2.loc[len(df2),['true_suvr_' + region,'synt_suvr']] = list(use[use.columns[:2]].iloc[0])
        

### Regional spatial correlation per subject (Figure 4 left panel)

In [ ]:
# select what to color plots by
totest = 'study'
totest2 = 'I-IV_true'

fs_res = np.array(dfs_fs)
spat_cols = ['file_names','spat_corr',totest,totest2]
spat_subj = pd.DataFrame(columns=spat_cols)
df_eval2 = df_eval.copy().reset_index(drop=True)

#create df with both regional spatial correlation and results of interest from df eval to color by
for subj,name,chrt,suvr in zip(fs_res,df_eval2['file_names'],df_eval2[totest],df_eval2[totest2]):
    true_spat = [v for v in subj[:,0] if not math.isnan(v)]
    synt_spat = [v for v in subj[:,1] if not math.isnan(v)]
    spat_subj.loc[len(spat_subj),spat_cols] = [name,
                                               np.corrcoef(true_spat,synt_spat)[0,1],
                                               chrt,
                                               suvr]




In [ ]:
# Create scatter plots for regional correlations as in Figure 4 in the article.
plt.rc('font', size=10)
figure, ax = plt.subplots(1,1,figsize=(3,4))
df_melted = spat_subj.melt(value_vars=['spat_corr'],var_name='variable',value_name='value')
df_melted2 = spat_subj.melt(value_vars=['spat_corr'],var_name='variable',value_name='value')
df_melted[totest] = pd.concat([spat_subj[totest]]).reset_index(drop=True)
df_melted2[totest2] = pd.concat([spat_subj[totest2]]).reset_index(drop=True)

# set size of scatter points and random seed to be consistent when plotting with different colormaps
if subsample:
    size = 4
    seed = 12
else:
    size = 3
    seed = 10

# select the colors per subject
cmap1 = plt.get_cmap('tab20b')
cmap2 = plt.get_cmap('tab20c')
if subsample:
    colors = [cmap2(0),cmap1(13),cmap1(0),cmap1(10),cmap2(9),cmap1(19)]
else:
    colors = [cmap2(0),cmap1(13),cmap1(0),cmap1(10),cmap2(9),cmap1(19),
             cmap2(6),cmap1(8),cmap2(17),cmap2(19),cmap2(11),cmap1(15),cmap1(1)]

# select colors for SUVR
suvrs_cbar=plt.Normalize(0.8,3.5)
sm = plt.cm.ScalarMappable(cmap=sns.cubehelix_palette(as_cmap=True),norm=suvrs_cbar)

#color for distribution
cols = ['grey']

categories = df_melted['variable'].astype('category').cat.codes
np.random.seed(seed)

#create the stripplots by categories
sns.stripplot(x=categories,y='value',data=df_melted,hue=totest,ax=ax, palette=colors,#palette='PRGn',#['white','lightsteelblue'],
            zorder=1,legend=False,size=size,edgecolor='black',linewidth=0.1,hue_order=hue_order,marker='p',
             jitter=0.3)
categories = df_melted2['variable'].astype('category').cat.codes
np.random.seed(seed)
sns.stripplot(x=categories+0.06,y='value',data=df_melted2,hue=totest2,ax=ax, palette=sns.cubehelix_palette(as_cmap=True),#palette='PRGn',#['white','lightsteelblue'],
            zorder=1,legend=False,size=size,edgecolor='black',linewidth=0.1,hue_order=hue_order,marker='p',
             jitter=0.3,hue_norm=suvrs_cbar)

#create distribution plot
sns.violinplot(x=categories+0.07,y='value',data=df_melted2,hue='variable',
               split=True,cut=0,ax=ax,bw_adjust=0.8,palette=cols,
               alpha=0.4,inner=None,zorder=0,legend=False,width=-0.8)

#plot configs
ax.axis([-0.8,2.8,-0.1,1])
ax.set_ylabel('R')
ax.set_xlabel('')
ax.set_xticks([])
ax.figure.colorbar(sm,ax=ax,shrink=0.4,location='right',label='Braak I-IV [SUVR]',aspect=10,pad=0.05,anchor=(0.0,0))

#save figure
if save_plots:
    plt.savefig('../outputs/figures/' + savename + '_subject.pdf',bbox_inches='tight')

### SUVR correlation per region (Figure 3 bottom panel)

In [ ]:
# dataframe for region, correlation, mean value (for color), lower and upper limits for confidence intervals
corrs = pd.DataFrame(columns=['region','corr','mean_suvr','lower','upper'])

# use both left and right regions
for reg,df in zip(rgs,dfs_reg):
    if reg.split('-')[1] == 'rh':
        regname = 'Right '
    else:
        regname = 'Left '
    regname = regname + reg.split('-')[2]
    
    # bootstrap for confidence intervals
    Rs = []
    for i in range(1000):
        idx_boot = np.random.choice(len(df),size=len(df),replace=True)
        df_boot = df.iloc[idx_boot]
        Rs.append(df_boot[['true_suvr_'+reg,'synt_suvr']].corr().iloc[0,1])
    Rs = sorted(Rs)
    lower = Rs[25]
    upper = Rs[975]
    
    corrs.loc[len(corrs)] = [regname,df[['true_suvr_'+reg,'synt_suvr']].corr().iloc[0,1],
                            df['true_suvr_'+reg].mean(),lower,upper]

In [ ]:
# Create bar plots for regional suvr correlations as in Figure 3 in the article.
plt.rc('font', size=12)
figure, ax = plt.subplots(1,1,figsize=(15,2.3))

#colormap and sorting
cmap=sns.cubehelix_palette(as_cmap=True)
corrs = corrs.sort_values(by='corr',ascending=False)
suvrs=plt.Normalize(1,1.4)
sm = plt.cm.ScalarMappable(cmap=cmap,norm=suvrs)

#plot correlations and confidence intervals
sns.barplot(corrs,y='corr',x='region',ax=ax,hue='mean_suvr',dodge=False,palette=cmap,hue_norm=suvrs)
ax.errorbar(corrs['region'],corrs['corr'],yerr=[corrs['corr']-corrs['lower'],corrs['upper']-corrs['corr']],
            fmt='none',ecolor='black',capsize=1,linewidth=0.4)

#plot configs
ax.axis([-2,len(rgs)+1,0,1])
ax.grid(alpha=0.2)
ax.tick_params(axis='x',labelsize=8,labelrotation=90)
ax.set_ylabel('R')
ax.set_xlabel('')
ax.get_legend().remove()
ax.figure.colorbar(sm,ax=ax,shrink=0.8,location='right',label='Average load [SUVR]',aspect=10,pad=0.02)

#save figure
if save_plots:
    plt.savefig('../outputs/figures/' + savename + '_suvr.pdf',bbox_inches='tight')

### Laterality index correlation per region (Figure 4 right panel)

In [ ]:
check = 'synt_suvr'
pos_lim = 1.36 #only compute laterality index in tau-positive individuals, select threshold for positivity

# dataframe for region, correlation, mean value (for color), lower and upper limits for confidence intervals
corrs = pd.DataFrame(columns=['region','corr','mean_suvr','lower','upper'])

# compare left and right regions
for reg_l, reg_r,df_l,df_r in zip(rgs[:34],rgs[34:],dfs_reg[:34],dfs_reg[34:]):
    df_l = df_l[(df_eval['I-IV_true'] > pos_lim).reset_index(drop=True)]
    df_r = df_r[(df_eval['I-IV_true'] > pos_lim).reset_index(drop=True)]
    true_li = 100*(df_l['true_suvr_'+reg_l]-df_r['true_suvr_'+reg_r])/((df_l['true_suvr_'+reg_l]+df_r['true_suvr_'+reg_r])/2)
    pred_li = 100*(df_l[check]-df_r[check])/((df_l[check]+df_r[check])/2)
    regname = reg_l.split('-')[2]
    
    # bootstrap for confidence intervals
    Rs = []
    for i in range(1000):
        idx_boot = np.random.choice(len(df_l),size=len(df_l),replace=True)
        true_li_boot = true_li.iloc[idx_boot]
        pred_li_boot = pred_li.iloc[idx_boot]
        Rs.append(np.corrcoef(list(true_li_boot),list(pred_li_boot))[0,1])
    Rs = sorted(Rs)
    lower = Rs[25]
    upper = Rs[975]

    corrs.loc[len(corrs)] = [regname,np.corrcoef(list(true_li),list(pred_li))[0,1],
                            true_li.abs().mean(),lower,upper]

In [ ]:
# Create bar plots for laterality index correlations as in Figure 4 in the article.
plt.rc('font', size=14)
figure, ax = plt.subplots(1,1,figsize=(4,6))

#colormap and sorting
cmap=sns.light_palette('steelblue',as_cmap=True)
corrs = corrs.sort_values(by='corr',ascending=False)
corrs['corr'] = corrs['corr'].abs()
suvrs=plt.Normalize(corrs['mean_suvr'].min(),corrs['mean_suvr'].max())
sm = plt.cm.ScalarMappable(cmap=cmap,norm=suvrs)

#plot correlations and confidence intervals
sns.barplot(corrs,x='corr',y='region',ax=ax,hue='mean_suvr',dodge=False,palette=cmap)
ax.errorbar(corrs['corr'],corrs['region'],xerr=[corrs['corr']-corrs['lower'],corrs['upper']-corrs['corr']],
            fmt='none',ecolor='black',capsize=1,linewidth=0.4)

#plot configs
ax.set_title('Laterality index',weight='bold',size=12)
ax.axis([0,1,len(rgs[:34])+1,-2])
ax.grid(alpha=0.2)
ax.set_ylabel('')
ax.tick_params(axis='y',labelsize=8)
ax.set_xlabel('R')
ax.set_xlabel('')
ax.get_legend().remove()
ax.figure.colorbar(sm,ax=ax,shrink=0.4,location='right',label='Average |LI|',aspect=10,pad=0.05,anchor=(0.0,0))

#save figure
if save_plots:
    plt.savefig('../outputs/figures/' + savename + '_suvr.pdf',bbox_inches='tight')